In [ ]:
import pandas as pd
import numpy as np
import json
import re
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Affichage
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

## 📥 Chargement des Données

In [ ]:
def load_jsonl(filepath):
    """Charge un fichier JSONL en DataFrame."""
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return data

# Chargement des données
train_data = load_jsonl('Data/train.jsonl')
test_data = load_jsonl('Data/kaggle_test.jsonl')

print(f"Train: {len(train_data)} tweets")
print(f"Test: {len(test_data)} tweets")

In [ ]:
# Examiner la structure d'un tweet
sample = train_data[0]
print("Clés disponibles:")
print(list(sample.keys()))
print("\nClés user:")
if 'user' in sample:
    print(list(sample['user'].keys()))

## 2.1 Features Textuelles Discriminantes

Les influenceurs ont tendance à:
- Utiliser des call-to-action (suivez, abonnez, etc.)
- Faire de l'auto-promotion (nouveau, vidéo, article)
- Utiliser beaucoup de hashtags et emojis
- Écrire des tweets plus longs

Les observateurs ont tendance à:
- Répondre aux autres (tweets commençant par @)
- Mentionner beaucoup de comptes
- Faire des RT/QT
- Écrire des tweets plus courts

In [ ]:
import emoji

def count_emojis(text):
    """Compte le nombre d'emojis dans un texte."""
    return len([c for c in text if c in emoji.EMOJI_DATA])

def get_text_content(tweet):
    """Extrait le texte complet du tweet."""
    if 'extended_tweet' in tweet and tweet['extended_tweet']:
        return tweet['extended_tweet'].get('full_text', tweet.get('text', ''))
    return tweet.get('text', '')

def extract_textual_features(tweet):
    """Extrait les features textuelles discriminantes."""
    text = get_text_content(tweet)
    text_lower = text.lower()
    
    features = {}
    
    # === Features Influenceurs ===
    
    # Call-to-action patterns
    cta_patterns = r'suivez|abonnez|cliquez|like|partage[zr]?|rt\s|retweet|subscribe|follow|share'
    features['has_call_to_action'] = 1 if re.search(cta_patterns, text_lower) else 0
    
    # Self-promotion patterns
    promo_patterns = r'nouveau|nouvelle|vidéo|article|podcast|live|thread|fil\s|lien|link|disponible'
    features['has_self_promotion'] = 1 if re.search(promo_patterns, text_lower) else 0
    
    # Hashtag count
    hashtags = re.findall(r'#\w+', text)
    features['hashtag_count'] = len(hashtags)
    features['is_hashtag_heavy'] = 1 if len(hashtags) > 3 else 0
    
    # Emoji count
    features['emoji_count'] = count_emojis(text)
    features['is_emoji_heavy'] = 1 if features['emoji_count'] > 5 else 0
    
    # Tweet length
    features['tweet_length'] = len(text)
    features['is_long_tweet'] = 1 if len(text) > 200 else 0
    features['word_count'] = len(text.split())
    
    # === Features Observateurs ===
    
    # Reply detection
    features['is_reply'] = 1 if text.startswith('@') else 0
    features['is_in_reply'] = 1 if tweet.get('in_reply_to_status_id') is not None else 0
    
    # Mention count
    mentions = re.findall(r'@\w+', text)
    features['mention_count'] = len(mentions)
    features['is_mention_heavy'] = 1 if len(mentions) > 2 else 0
    
    # Quote/RT detection
    features['has_rt_qt'] = 1 if re.search(r'RT @|QT:|\brt\b', text) else 0
    features['is_quote_status'] = 1 if tweet.get('is_quote_status', False) else 0
    
    # Short tweet
    features['is_short_tweet'] = 1 if len(text) < 100 else 0
    
    # === Features Textuelles Additionnelles ===
    
    # URL count
    urls = re.findall(r'https?://\S+', text)
    features['url_count'] = len(urls)
    features['has_url'] = 1 if len(urls) > 0 else 0
    
    # Uppercase ratio (crier = engagement?)
    letters = [c for c in text if c.isalpha()]
    if len(letters) > 0:
        features['uppercase_ratio'] = sum(1 for c in letters if c.isupper()) / len(letters)
    else:
        features['uppercase_ratio'] = 0
    
    # Punctuation
    features['exclamation_count'] = text.count('!')
    features['question_count'] = text.count('?')
    features['has_multiple_exclamations'] = 1 if text.count('!') > 1 else 0
    
    # First person pronouns (je, mon, ma, mes, moi)
    first_person = r'\bje\b|\bmon\b|\bma\b|\bmes\b|\bmoi\b|\bj\'|\bm\''
    features['first_person_count'] = len(re.findall(first_person, text_lower))
    
    # Média references (photo, image, vidéo)
    media_patterns = r'photo|image|vidéo|video|pic\b'
    features['has_media_reference'] = 1 if re.search(media_patterns, text_lower) else 0
    
    return features

# Test sur un tweet
sample_features = extract_textual_features(train_data[0])
print("Exemple de features textuelles:")
for k, v in sample_features.items():
    print(f"  {k}: {v}")

## 2.2 Features Structurées du Tweet

In [ ]:
def parse_source(source_html):
    """Parse le champ source pour extraire le client Twitter."""
    if not source_html:
        return 'unknown'
    match = re.search(r'>([^<]+)<', source_html)
    if match:
        return match.group(1)
    return 'unknown'

def extract_structured_features(tweet):
    """Extrait les features structurées du tweet."""
    features = {}
    
    # === Tweet metadata ===
    
    # Source (device)
    source = parse_source(tweet.get('source', ''))
    features['source_raw'] = source
    features['is_iphone'] = 1 if 'iPhone' in source else 0
    features['is_android'] = 1 if 'Android' in source else 0
    features['is_web'] = 1 if 'Web' in source else 0
    features['is_tweetdeck'] = 1 if 'TweetDeck' in source else 0
    features['is_bot_source'] = 1 if any(x in source.lower() for x in ['bot', 'hootsuite', 'buffer', 'dlvr', 'ifttt', 'zapier']) else 0
    
    # Engagement metrics
    features['retweet_count'] = tweet.get('retweet_count', 0) or 0
    features['favorite_count'] = tweet.get('favorite_count', 0) or 0
    features['reply_count'] = tweet.get('reply_count', 0) or 0
    features['quote_count'] = tweet.get('quote_count', 0) or 0
    
    # Total engagement
    features['total_engagement'] = (
        features['retweet_count'] + 
        features['favorite_count'] + 
        features['reply_count'] + 
        features['quote_count']
    )
    
    # Log transforms for engagement (reduce skewness)
    features['log_retweet_count'] = np.log1p(features['retweet_count'])
    features['log_favorite_count'] = np.log1p(features['favorite_count'])
    features['log_total_engagement'] = np.log1p(features['total_engagement'])
    
    # Reply/Quote status
    features['is_reply_to_someone'] = 1 if tweet.get('in_reply_to_user_id') is not None else 0
    features['is_quote_status'] = 1 if tweet.get('is_quote_status', False) else 0
    
    # Has quoted status
    features['has_quoted_status'] = 1 if 'quoted_status' in tweet and tweet['quoted_status'] else 0
    
    # Entities counts from entities field
    entities = tweet.get('entities', {})
    features['entities_hashtags'] = len(entities.get('hashtags', []))
    features['entities_urls'] = len(entities.get('urls', []))
    features['entities_mentions'] = len(entities.get('user_mentions', []))
    features['entities_symbols'] = len(entities.get('symbols', []))
    
    return features

# Test
sample_struct = extract_structured_features(train_data[0])
print("Exemple de features structurées:")
for k, v in sample_struct.items():
    print(f"  {k}: {v}")

In [ ]:
def extract_user_features(tweet):
    """Extrait les features de l'utilisateur."""
    features = {}
    user = tweet.get('user', {})
    
    if not user:
        return {k: 0 for k in [
            'user_followers_count', 'user_friends_count', 'user_statuses_count',
            'user_favourites_count', 'user_listed_count', 'user_verified',
            'user_followers_friends_ratio', 'user_avg_tweets_per_day',
            'user_description_length', 'user_has_url', 'user_default_profile',
            'user_default_profile_image', 'user_geo_enabled',
            'log_user_followers', 'log_user_friends', 'log_user_statuses'
        ]}
    
    # Basic counts
    features['user_followers_count'] = user.get('followers_count', 0) or 0
    features['user_friends_count'] = user.get('friends_count', 0) or 0
    features['user_statuses_count'] = user.get('statuses_count', 0) or 0
    features['user_favourites_count'] = user.get('favourites_count', 0) or 0
    features['user_listed_count'] = user.get('listed_count', 0) or 0
    
    # Verified status
    features['user_verified'] = 1 if user.get('verified', False) else 0
    
    # Ratios
    friends = features['user_friends_count']
    if friends > 0:
        features['user_followers_friends_ratio'] = features['user_followers_count'] / friends
    else:
        features['user_followers_friends_ratio'] = features['user_followers_count']
    
    # Account age and activity (approximation via statuses count)
    features['user_avg_tweets_per_day'] = 0  # Would need account creation date
    
    # Profile features
    description = user.get('description', '') or ''
    features['user_description_length'] = len(description)
    features['user_has_url'] = 1 if user.get('url') else 0
    features['user_default_profile'] = 1 if user.get('default_profile', False) else 0
    features['user_default_profile_image'] = 1 if user.get('default_profile_image', False) else 0
    features['user_geo_enabled'] = 1 if user.get('geo_enabled', False) else 0
    
    # Log transforms
    features['log_user_followers'] = np.log1p(features['user_followers_count'])
    features['log_user_friends'] = np.log1p(features['user_friends_count'])
    features['log_user_statuses'] = np.log1p(features['user_statuses_count'])
    
    return features

# Test
sample_user = extract_user_features(train_data[0])
print("Exemple de features utilisateur:")
for k, v in sample_user.items():
    print(f"  {k}: {v}")

## 2.3 Features NLP Avancées

In [ ]:
# Installation des dépendances si nécessaire
# !pip install transformers torch

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch

# Vérifier si GPU disponible
device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'GPU' if device == 0 else 'CPU'}")

# Sentiment Analysis - modèle multilingue
print("Loading sentiment model...")
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    device=device,
    truncation=True,
    max_length=512
)
print("Sentiment model loaded!")

In [ ]:
def get_sentiment_features(text, sentiment_pipe):
    """Extrait les features de sentiment."""
    try:
        result = sentiment_pipe(text[:512])[0]  # Truncate to 512 chars
        # Le modèle retourne 1-5 stars
        label = result['label']  # e.g., "4 stars"
        score = result['score']
        stars = int(label.split()[0])
        
        return {
            'sentiment_stars': stars,
            'sentiment_score': score,
            'sentiment_positive': 1 if stars >= 4 else 0,
            'sentiment_negative': 1 if stars <= 2 else 0,
            'sentiment_neutral': 1 if stars == 3 else 0
        }
    except Exception as e:
        return {
            'sentiment_stars': 3,
            'sentiment_score': 0.5,
            'sentiment_positive': 0,
            'sentiment_negative': 0,
            'sentiment_neutral': 1
        }

# Test
test_text = get_text_content(train_data[0])
print(f"Text: {test_text[:100]}...")
print(f"Sentiment: {get_sentiment_features(test_text, sentiment_pipeline)}")

In [ ]:
# NER - Named Entity Recognition français
print("Loading NER model...")
try:
    ner_pipeline = pipeline(
        "ner",
        model="Jean-Baptiste/camembert-ner",
        device=device,
        aggregation_strategy="simple"
    )
    print("NER model loaded!")
    NER_AVAILABLE = True
except Exception as e:
    print(f"NER model not available: {e}")
    NER_AVAILABLE = False

In [ ]:
def get_ner_features(text, ner_pipe):
    """Extrait les features NER."""
    try:
        entities = ner_pipe(text[:512])
        entity_counts = Counter([e['entity_group'] for e in entities])
        
        return {
            'ner_person_count': entity_counts.get('PER', 0),
            'ner_org_count': entity_counts.get('ORG', 0),
            'ner_loc_count': entity_counts.get('LOC', 0),
            'ner_misc_count': entity_counts.get('MISC', 0),
            'ner_total_entities': len(entities),
            'ner_has_person': 1 if entity_counts.get('PER', 0) > 0 else 0,
            'ner_has_org': 1 if entity_counts.get('ORG', 0) > 0 else 0
        }
    except Exception as e:
        return {
            'ner_person_count': 0,
            'ner_org_count': 0,
            'ner_loc_count': 0,
            'ner_misc_count': 0,
            'ner_total_entities': 0,
            'ner_has_person': 0,
            'ner_has_org': 0
        }

if NER_AVAILABLE:
    print(f"NER: {get_ner_features(test_text, ner_pipeline)}")

## 🔄 Extraction Complète des Features

In [ ]:
from tqdm import tqdm

def extract_all_features(tweet, sentiment_pipe=None, ner_pipe=None):
    """Extrait toutes les features d'un tweet."""
    features = {}
    
    # ID et label
    features['challenge_id'] = tweet.get('challenge_id')
    features['label'] = tweet.get('label')
    
    # Texte
    text = get_text_content(tweet)
    features['text'] = text
    
    # Features textuelles
    features.update(extract_textual_features(tweet))
    
    # Features structurées
    features.update(extract_structured_features(tweet))
    
    # Features utilisateur
    features.update(extract_user_features(tweet))
    
    # Features NLP (optionnel - plus lent)
    if sentiment_pipe:
        features.update(get_sentiment_features(text, sentiment_pipe))
    
    if ner_pipe and NER_AVAILABLE:
        features.update(get_ner_features(text, ner_pipe))
    
    return features

In [ ]:
# Extraction des features de base (sans NLP avancé - plus rapide)
print("Extracting basic features for train set...")
train_features = []
for tweet in tqdm(train_data):
    train_features.append(extract_all_features(tweet))

train_df = pd.DataFrame(train_features)
print(f"Train features shape: {train_df.shape}")
train_df.head()

In [ ]:
print("Extracting basic features for test set...")
test_features = []
for tweet in tqdm(test_data):
    test_features.append(extract_all_features(tweet))

test_df = pd.DataFrame(test_features)
print(f"Test features shape: {test_df.shape}")
test_df.head()

## 📊 Analyse des Features

In [ ]:
# Corrélation avec le label
numeric_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['challenge_id', 'label']]

correlations = train_df[numeric_cols + ['label']].corr()['label'].drop('label').sort_values(ascending=False)
print("Top 20 features les plus corrélées avec le label (Influencer=1):")
print(correlations.head(20))
print("\n... et les moins corrélées:")
print(correlations.tail(20))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualisation
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Distribution des features clés par label
features_to_plot = ['log_user_followers', 'log_total_engagement', 'hashtag_count', 'mention_count']

for ax, feat in zip(axes.flatten(), features_to_plot):
    for label in [0, 1]:
        data = train_df[train_df['label'] == label][feat]
        label_name = 'Influencer' if label == 1 else 'Observer'
        ax.hist(data, bins=50, alpha=0.5, label=label_name, density=True)
    ax.set_title(f'Distribution de {feat}')
    ax.legend()

plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150)
plt.show()

In [ ]:
# Comparaison moyennes par classe
comparison = train_df.groupby('label')[numeric_cols].mean().T
comparison.columns = ['Observer', 'Influencer']
comparison['Diff'] = comparison['Influencer'] - comparison['Observer']
comparison['Diff_pct'] = (comparison['Diff'] / (comparison['Observer'].abs() + 1e-8)) * 100

print("Différences moyennes entre Influencers et Observers:")
comparison.sort_values('Diff_pct', ascending=False).head(20)

## 💾 Sauvegarde des Features

In [ ]:
# Sauvegarder les features extraites
train_df.to_csv('Data/train_features.csv', index=False)
test_df.to_csv('Data/test_features.csv', index=False)

print(f"Features sauvegardées!")
print(f"  - Train: Data/train_features.csv ({train_df.shape})")
print(f"  - Test: Data/test_features.csv ({test_df.shape})")

## 🧪 Test Rapide avec LightGBM

In [ ]:
from sklearn.model_selection import cross_val_score
from lightgbm import LGBMClassifier

# Préparer les données
feature_cols = [c for c in numeric_cols if c not in ['challenge_id', 'label']]
X = train_df[feature_cols].fillna(0)
y = train_df['label']

print(f"Features utilisées: {len(feature_cols)}")
print(f"X shape: {X.shape}")

# Model simple
lgbm = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=7,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

# Cross-validation
scores = cross_val_score(lgbm, X, y, cv=5, scoring='accuracy')
print(f"\nCross-validation Accuracy: {scores.mean():.4f} (+/- {scores.std()*2:.4f})")

In [ ]:
# Feature importance
lgbm.fit(X, y)
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': lgbm.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 20 features les plus importantes:")
print(importance.head(20))

# Plot
plt.figure(figsize=(10, 8))
plt.barh(importance.head(20)['feature'], importance.head(20)['importance'])
plt.xlabel('Importance')
plt.title('Top 20 Feature Importance (LightGBM)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

## 🚀 Génération des Prédictions

In [ ]:
# Préparer test set
X_test = test_df[feature_cols].fillna(0)

# Prédictions
predictions = lgbm.predict(X_test)

# Créer submission
submission = pd.DataFrame({
    'id': test_df['challenge_id'],
    'label': predictions
})

submission.to_csv('submission_features.csv', index=False)
print(f"Submission sauvegardée: submission_features.csv")
print(f"Distribution des prédictions:")
print(submission['label'].value_counts(normalize=True))

## ✅ Résumé Phase 2

### Features Extraites:

**Textuelles (22 features):**
- Call-to-action, self-promotion patterns
- Hashtags, emojis, mentions counts
- Tweet length, word count
- Reply/quote status
- URLs, uppercase ratio, punctuation

**Structurées (20 features):**
- Source (device)
- Engagement metrics (retweet, favorite, reply, quote counts)
- Log-transformed engagement
- Entities counts

**Utilisateur (15 features):**
- Followers, friends, statuses counts
- Verified status
- Followers/friends ratio
- Profile features

**NLP (optionnel, 12 features):**
- Sentiment (stars, positive/negative/neutral)
- NER (person, org, location counts)

### Prochaines étapes:
- Combiner avec TF-IDF (baseline.ipynb)
- Utiliser dans ensemble/stacking
- Fine-tuner les hyperparamètres